In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
import pytorch_lightning as pl
from torch.utils.data import Dataset, Subset, DataLoader

from hiermoe.constants import *
from hiermoe.extra_functions import set_seed
from hiermoe.data_setup import ImageDataset, HierImageDataset
from hiermoe.hierarchy import Hierarchy
from hiermoe import Trainer, HierMoeNet

In [3]:
set_seed(SEED)

DataPath = '/Users/alexandermichaeltjhin/Everything/Repos/Zooplankton/data/WHOI-Plankton'
DataSubdirectory = [
    '2006','2007','2008','2009','2010','2011', '2012',
    '2013','2014']
adj_graph = whoi_adjacency_graph_s
RESOLUTION = 65

SELECTED_LIST = [key for key, item in adj_graph.items() if len(item) == 0]

dataset = ImageDataset(
    data_directory = '/Users/alexandermichaeltjhin/Everything/Repos/Zooplankton/data/WHOI-Plankton',
    data_subdirectories = DataSubdirectory,
    class_names = SELECTED_LIST,
    max_class_size = 10000,
    image_resolution = 64,
    image_transforms = None,
    format_file = '.png',
    seed = SEED
    )

whoi_dataset = HierImageDataset(
    base_dataset=dataset,
    adjacency_graph = whoi_adjacency_graph_s,
    levels=3,
    leaves_only=True
)
whoi_dataset.print_dataset_details()

[leaves_only] Kept 70843 samples | Removed 0 non-leaf samples

Total Dataset: Size = 70843 | Levels = 3
all_node_counts: {0: 70843, 1: 54338, 3: 12148, 7: 10000, 8: 295, 9: 1853, 4: 42190, 10: 0, 11: 10000, 12: 2191, 13: 10000, 14: 0, 15: 10000, 16: 0, 17: 9999, 2: 16505, 5: 5161, 18: 0, 19: 5161, 6: 11344, 20: 10000, 21: 711, 22: 633}

nodes_by_level: {0: [0], 1: [1, 2], 2: [3, 4, 5, 6], 3: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]}


------------------------Level 0------------------------
Level: 0 | Class Name: root                 | Class Label:   0 | Type: Parent | Count:  70843 | Prop: 1.00

------------------------Level 1------------------------
Level: 1 | Class Name: Colonial             | Class Label:   1 | Type: Parent | Count:  54338 | Prop: 0.77
Level: 1 | Class Name: Unicellular          | Class Label:   2 | Type: Parent | Count:  16505 | Prop: 0.23

------------------------Level 2------------------------
Level: 2 | Class Name: C-Spines             | Cla

In [4]:
from torch.utils.data import Dataset, Subset, DataLoader, SequentialSampler, WeightedRandomSampler, random_split
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.Pad(padding = 5, fill = 0),
    transforms.Resize((RESOLUTION, RESOLUTION)),
    transforms.ToTensor(),
])


whoi_dataset.append_image_transforms(
    image_transforms = train_transforms, replace = True
)

TRAIN_PROP = 0.7
VAL_PROP = 0.1
TEST_PROP = 0.2

BATCH_SIZE = 64

train_split, val_split, test_split = whoi_dataset.split_train_test_val(
    train_prop = TRAIN_PROP, val_prop = VAL_PROP, test_prop = TEST_PROP
)

# Create dataloaders
train_loader, val_loader, test_loader = whoi_dataset.create_dataloaders(
    batch_size = BATCH_SIZE,
    train_indices = train_split,
    val_indices = val_split,
    test_indices = test_split,
    image_transforms = None,
    train_sample_weights = None
)


In [11]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'Using device: MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using device: CUDA GPU')
else:
    device = torch.device('cpu')
    print(f'Using device: CPU')

model = HierMoeNet(whoi_dataset.hierarchy, whoi_dataset.label_to_ids)

Using device: MPS (Apple Silicon GPU)


In [12]:
import time
from datetime import date



today = date.today()
HYPERPARAMETERS = {
    'lr': 2e-4, 
    'epochs': 20, 
    'scheduler':{'type': 'CosineAnnealingLR', 'T_max': 50},
    'early_stopping': {'patience': 3, 'delta': 0.001},
}
start = time.time()
trainer = Trainer(learning_rate=HYPERPARAMETERS['lr'], max_epochs=HYPERPARAMETERS['epochs'], device=device,
                  print_every=1, model_dir=f"efficient_net_whoi_{today}")
trainer.fit(model, train_loader, val_loader, 
            patience=HYPERPARAMETERS['early_stopping']['patience'],
            delta=HYPERPARAMETERS['early_stopping']['delta'])
saved_dir = trainer.model_dir
print(f"Total time taken {round((time.time() - start) / 60, 2)} minutes")

Training at device: mps
beginning training
Epoch [1/20] | Train Loss: 0.1369 | Train Acc: 0.4803 | Train F1: 0.3334 | Valid Loss: 0.1130 | Valid Acc: 0.5892 | Valid F1: 0.4599 | Saved best model (val_acc=0.5892)
Epoch [2/20] | Train Loss: 0.1108 | Train Acc: 0.5999 | Train F1: 0.4835 | Valid Loss: 0.1029 | Valid Acc: 0.6331 | Valid F1: 0.5458 | Saved best model (val_acc=0.6331)
Epoch [3/20] | Train Loss: 0.1013 | Train Acc: 0.6445 | Train F1: 0.5380 | Valid Loss: 0.0914 | Valid Acc: 0.6820 | Valid F1: 0.5957 | Saved best model (val_acc=0.6820)
Epoch [4/20] | Train Loss: 0.0921 | Train Acc: 0.6842 | Train F1: 0.5854 | Valid Loss: 0.0863 | Valid Acc: 0.7085 | Valid F1: 0.6184 | Saved best model (val_acc=0.7085)
Epoch [5/20] | Train Loss: 0.0852 | Train Acc: 0.7105 | Train F1: 0.6192 | Valid Loss: 0.0955 | Valid Acc: 0.6798 | Valid F1: 0.5871
Epoch [6/20] | Train Loss: 0.0798 | Train Acc: 0.7325 | Train F1: 0.6461 | Valid Loss: 0.0729 | Valid Acc: 0.7558 | Valid F1: 0.6878 | Saved best mo

In [13]:
trainer.model_dir

'efficient_net_whoi_2026-03-11'

In [14]:
# saved_dir = "efficient_net_whoi_2026-03-11_2"
model = HierMoeNet(whoi_dataset.hierarchy, whoi_dataset.label_to_ids, checkpoint_dir=saved_dir)
# trainer = Trainer(learning_rate=HYPERPARAMETERS['lr'], max_epochs=HYPERPARAMETERS['epochs'], device=device,
#                   print_every=1, model_dir=trainer.model_dir)
result = trainer.predict(model, test_loader, save=True)

Loaded checkpoint: efficient_net_whoi_2026-03-11/best_model.pt
Hierarchical Evaluation:
  Level 1: Acc=0.9587 | F1=0.9432 | Prec=0.9346 | Rec=0.9528 | n=14168
    Colonial               Acc=0.9638 | F1=0.9729 | Prec=0.9821 | Rec=0.9638 | n=10884
    Unicellular            Acc=0.9418 | F1=0.9136 | Prec=0.8870 | Rec=0.9418 | n=3284
  Level 2: Acc=0.9175 | F1=0.9015 | Prec=0.8912 | Rec=0.9130 | n=14168
    C-Spines               Acc=0.8434 | F1=0.8509 | Prec=0.8585 | Rec=0.8434 | n=2439
    C-NoSpines             Acc=0.9311 | F1=0.9400 | Prec=0.9491 | Rec=0.9311 | n=8445
    U-Spines               Acc=0.9419 | F1=0.9025 | Prec=0.8663 | Rec=0.9419 | n=1032
    U-NoSpines             Acc=0.9356 | F1=0.9127 | Prec=0.8909 | Rec=0.9356 | n=2252
  Level 3: Acc=0.8411 | F1=0.7853 | Prec=0.8104 | Rec=0.7807 | n=14168
    Chaetoceros            Acc=0.8058 | F1=0.8265 | Prec=0.8483 | Rec=0.8058 | n=2019
    Lauderia               Acc=0.2105 | F1=0.3158 | Prec=0.6316 | Rec=0.2105 | n=57
    Asterion

In [16]:
from hiermoe import Visualize

In [17]:
vis = Visualize(trainer.model_dir)
vis.plot_train()

NameError: name 'Path' is not defined